# Unsupervised Graph Autoencoder (GAE) for Log Anomaly Detection

This notebook implements an **Attribute-Aware Variational Graph Autoencoder (VGAE)** or standard GAE to detect anomalies in execution log sequences purely in an unsupervised manner.

### Approach
- **Encoder**: Learns structural and temporal log representations using `GINEConv`.
- **Multi-Task Decoder**: Reconstructs Graph Structure (Edges), Node Features (Semantics & Position), and Edge Attributes (Time-Deltas).
- **Anomaly Scoring**: Graphs with high reconstruction error (across any of the 3 components) are flagged as anomalies.
- **Training Paradigm**: Clean-Train (trained only on normal graphs `y=0`) vs. Noisy-Train (trained on all data regardless of label).

In [ ]:
import os
import time
import json
from pathlib import Path
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam

from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINEConv
from torch_geometric.utils import negative_sampling, scatter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.metrics import precision_recall_curve, roc_auc_score, auc, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Setup & Configuration

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
RUN_TAG = "20260407_2026"
DATA_DIR = Path("../data/processed/")
GRAPH_DATASET_PATH = DATA_DIR / f"{RUN_TAG}_graph_dataset.pt"

# TEST RUN SETTINGS
TEST_RUN = False          # Set to True to use a small subset of data for quick learning verification
TEST_SAMPLES = 5000      # Number of graphs to use in the test run

# Hyperparameters
HIDDEN_DIM = 128
LATENT_DIM = 64
BATCH_SIZE = 256
EPOCHS = 25
LEARNING_RATE = 1e-2

# Multi-Task Loss Weights (Normalized inputs allow us to use balanced weights)
ALPHA = 1.0     # Structure Reconstruction Weight
BETA = 1.0      # Node Feature Reconstruction Weight 
GAMMA = 1.0     # Edge Attribute (Time-Delta) Reconstruction Weight

TRAIN_MODE = "clean"  # "clean": normal data only, "noisy": mixed data

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {DEVICE}")
print(f"Test Run Mode: {TEST_RUN} (Samples: {TEST_SAMPLES}, Epochs: {EPOCHS})")


## 2. Load Dataset & Prepare Splits

In [ ]:
# Load the generated graph dataset
print(f"Loading dataset from {GRAPH_DATASET_PATH}...")
dataset = torch.load(GRAPH_DATASET_PATH, weights_only=False)

all_data  = dataset["data_list"]
idx_train = dataset["idx_train"]
idx_val   = dataset["idx_val"]
idx_test  = dataset["idx_test"]
NODE_DIM  = dataset["node_dim"]
EDGE_DIM  = dataset["edge_dim"]

print(f"Total Graphs: {len(all_data):,}")
print(f"Node Dim: {NODE_DIM}, Edge Dim: {EDGE_DIM}")

# Filter Train Set based on TRAIN_MODE
train_graphs = [all_data[i] for i in idx_train]

if TRAIN_MODE == "clean":
    # Keep only normal logs for training
    train_graphs = [g for g in train_graphs if g.y.item() == 0]
    print(f"Clean-Train: Filtered out anomalies. Training on {len(train_graphs):,} normal graphs.")
else:
    print(f"Noisy-Train: Training on all {len(train_graphs):,} graphs (unsupervised mixed).")

# TEST RUN SUBSAMPLING
if TEST_RUN:
    train_graphs = train_graphs[:TEST_SAMPLES]
    print(f"--- TEST RUN: Subsampled training set to {len(train_graphs)} graphs ---")

# Validation and Test sets remain mixed to test anomaly detection
# Optionally subsample validation/test for speed during TEST_RUN
if TEST_RUN:
    val_graphs  = [all_data[i] for i in idx_val][:TEST_SAMPLES]
    test_graphs = [all_data[i] for i in idx_test][:TEST_SAMPLES]
else:
    val_graphs  = [all_data[i] for i in idx_val]
    test_graphs = [all_data[i] for i in idx_test]

train_loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_graphs,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_graphs,  batch_size=BATCH_SIZE)


## 3. Define Attribute-Aware GAE Model

In [ ]:
class AttributeAwareGAE(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_dim=128, latent_dim=64):
        super().__init__()
        
        # --- INPUT STANDARDIZATION ---
        # Track running mean and std of raw features to force inputs to N(0, 1)
        self.raw_node_norm = nn.BatchNorm1d(node_dim, affine=False)
        self.raw_edge_norm = nn.BatchNorm1d(edge_dim, affine=False)

        # --- ENCODER ---
        self.node_proj = nn.Linear(node_dim, hidden_dim)
        self.edge_proj = nn.Linear(edge_dim, hidden_dim)
        
        # GINE layer incorporates edge features into message passing
        nn1 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim)
        )
        self.encoder_conv = GINEConv(nn1, edge_dim=hidden_dim)
        
        # --- DECODERS ---
        # 1. Structure Decoder is handled via inner product: Z_i * Z_j
        
        # 2. Node Feature Decoder
        self.node_decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, node_dim)
        )
        
        # 3. Edge Attribute Decoder
        self.edge_decoder = nn.Sequential(
            nn.Linear(latent_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, edge_dim)
        )

    def standardize_inputs(self, x, edge_attr):
        # Normalizes raw inputs so we don't get exploding MSE loss targets
        x_norm = self.raw_node_norm(x)
        
        if edge_attr is not None and edge_attr.size(0) > 0:
            if edge_attr.dim() == 1:
                edge_attr = edge_attr.unsqueeze(1)
            try:
                edge_attr_norm = self.raw_edge_norm(edge_attr)
            except ValueError:
                edge_attr_norm = edge_attr # Fallback if batch size is 1
        else:
            edge_attr_norm = edge_attr
            
        return x_norm, edge_attr_norm

    def encode(self, x_norm, edge_index, edge_attr_norm):
        x_h = self.node_proj(x_norm)
        edge_attr_h = self.edge_proj(edge_attr_norm) if edge_attr_norm is not None else None
        z = self.encoder_conv(x_h, edge_index, edge_attr_h)
        return z

    def decode_structure(self, z, edge_index):
        # Return raw logits for BCEWithLogitsLoss
        src, dst = edge_index
        return (z[src] * z[dst]).sum(dim=1)

    def decode_node_features(self, z):
        return self.node_decoder(z)
        
    def decode_edge_attributes(self, z, edge_index):
        src, dst = edge_index
        # Concatenate source and destination latent vectors
        z_edge = torch.cat([z[src], z[dst]], dim=-1)
        return self.edge_decoder(z_edge)
        
    def forward(self, x, edge_index, edge_attr):
        x_norm, edge_attr_norm = self.standardize_inputs(x, edge_attr)
        z = self.encode(x_norm, edge_index, edge_attr_norm)
        return z, x_norm, edge_attr_norm


## 4. Multi-Task Training Loop

In [ ]:
def train_step(model, loader, optimizer):
    model.train()
    total_loss, total_str_loss, total_node_loss, total_edge_loss = 0, 0, 0, 0
    
    for batch in tqdm(loader, desc="Training", leave=False):
        batch = batch.to(DEVICE)
        optimizer.zero_grad()
        
        # Predict the STANDARDIZED inputs
        z, x_norm, edge_attr_norm = model(batch.x, batch.edge_index, batch.edge_attr)
        
        # 1. Structure Loss (Positive & Negative Edges)
        pos_logits = model.decode_structure(z, batch.edge_index)
        neg_edge_index = negative_sampling(batch.edge_index, num_nodes=batch.num_nodes, num_neg_samples=batch.edge_index.size(1))
        neg_logits = model.decode_structure(z, neg_edge_index)
        
        loss_pos = F.binary_cross_entropy_with_logits(pos_logits, torch.ones_like(pos_logits))
        loss_neg = F.binary_cross_entropy_with_logits(neg_logits, torch.zeros_like(neg_logits))
        loss_str = loss_pos + loss_neg
        
        # 2. Node Feature Loss (MSE against NORMALIZED inputs)
        x_rec = model.decode_node_features(z)
        loss_node = F.mse_loss(x_rec, x_norm) 
        
        # 3. Edge Attribute Loss
        loss_edge = torch.tensor(0.0, device=DEVICE)
        if edge_attr_norm is not None and edge_attr_norm.size(0) > 0:
            edge_rec = model.decode_edge_attributes(z, batch.edge_index)
            loss_edge = F.mse_loss(edge_rec, edge_attr_norm)
            
        # Total Weighted Loss
        loss = ALPHA * loss_str + BETA * loss_node + GAMMA * loss_edge
        
        loss.backward()
        
        # Gradient Clipping to prevent explosion
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item() * batch.num_graphs
        total_str_loss += loss_str.item() * batch.num_graphs
        total_node_loss += loss_node.item() * batch.num_graphs
        total_edge_loss += loss_edge.item() * batch.num_graphs
        
    num_graphs = len(loader.dataset)
    if num_graphs == 0: return 0, 0, 0, 0
    return total_loss / num_graphs, total_str_loss / num_graphs, total_node_loss / num_graphs, total_edge_loss / num_graphs

# Initialize Model
model = AttributeAwareGAE(NODE_DIM, EDGE_DIM, HIDDEN_DIM, LATENT_DIM).to(DEVICE)
optimizer = Adam(model.parameters(), lr=LEARNING_RATE)

# Loss History for Plotting
history = {'total': [], 'structure': [], 'node': [], 'edge': []}

# Training Execution
print("Starting Training...")
print(f"Weights -> ALPHA (Str): {ALPHA}, BETA (Node): {BETA}, GAMMA (Edge): {GAMMA}")
for epoch in range(1, EPOCHS + 1):
    l_tot, l_str, l_nod, l_edg = train_step(model, train_loader, optimizer)
    history['total'].append(l_tot)
    history['structure'].append(l_str)
    history['node'].append(l_nod)
    history['edge'].append(l_edg)
    print(f"Epoch {epoch:02d} | Loss: {l_tot:.4f} (Str: {l_str:.4f}, Node: {l_nod:.4f}, Edge: {l_edg:.4f})")


## 5. Inference & Validation

To validate, we calculate the batched graph-level anomaly scores. The anomaly score is the combination of reconstruction errors. We then find the optimal threshold using a Precision-Recall Curve.

In [ ]:
def evaluate_anomaly_scores(model, loader):
    model.eval()
    all_scores, all_labels = [], []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating", leave=False):
            batch = batch.to(DEVICE)
            z, x_norm, edge_attr_norm = model(batch.x, batch.edge_index, batch.edge_attr)
            
            num_graphs = batch.num_graphs if hasattr(batch, 'num_graphs') else 1
            
            # 1. Structure Error (Graph level aggregation)
            pos_logits = model.decode_structure(z, batch.edge_index)
            pos_probs = torch.sigmoid(pos_logits)
            edge_errors = F.binary_cross_entropy(pos_probs, torch.ones_like(pos_probs), reduction='none')
            
            edge_batch = batch.batch[batch.edge_index[0]] if hasattr(batch, 'batch') and batch.batch is not None else torch.zeros(batch.edge_index.size(1), dtype=torch.long, device=DEVICE)
            graph_str_error = scatter(edge_errors, edge_batch, dim=0, reduce='mean', dim_size=num_graphs)
            
            # 2. Node Error (MSE against NORMALIZED inputs)
            x_rec = model.decode_node_features(z)
            node_errors = F.mse_loss(x_rec, x_norm, reduction='none').mean(dim=1)
            batch_idx = batch.batch if hasattr(batch, 'batch') and batch.batch is not None else torch.zeros(batch.x.size(0), dtype=torch.long, device=DEVICE)
            graph_node_error = scatter(node_errors, batch_idx, dim=0, reduce='mean', dim_size=num_graphs)
            
            # 3. Edge Attribute Error (MSE against NORMALIZED inputs)
            if edge_attr_norm is not None and edge_attr_norm.size(0) > 0:
                edge_rec = model.decode_edge_attributes(z, batch.edge_index)
                edge_attr_errors = F.mse_loss(edge_rec, edge_attr_norm, reduction='none').mean(dim=1)
                graph_edge_error = scatter(edge_attr_errors, edge_batch, dim=0, reduce='mean', dim_size=num_graphs)
            else:
                graph_edge_error = torch.zeros_like(graph_str_error)
            
            # Fallbacks in case graph_str_error or graph_edge_error contains NaN
            graph_str_error = torch.nan_to_num(graph_str_error, 0.0)
            graph_edge_error = torch.nan_to_num(graph_edge_error, 0.0)

            # Total Score
            total_scores = ALPHA * graph_str_error + BETA * graph_node_error + GAMMA * graph_edge_error
            
            all_scores.append(total_scores.cpu())
            all_labels.append(batch.y.cpu())
            
    return torch.cat(all_scores).numpy(), torch.cat(all_labels).numpy()

print("Evaluating on Validation Set...")
val_scores, val_labels = evaluate_anomaly_scores(model, val_loader)

# Find Optimal Threshold via PR-Curve
precision, recall, thresholds = precision_recall_curve(val_labels, val_scores)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print(f"\nValidation Complete!")
print(f"Best Threshold: {best_threshold:.4f}")
print(f"Best Val F1-Score: {f1_scores[best_idx]:.4f}")
print(f"Val PR-AUC: {auc(recall, precision):.4f}")
print(f"Val ROC-AUC: {roc_auc_score(val_labels, val_scores):.4f}")


## 6. Test Evaluation

Using the threshold discovered on the validation set, we evaluate the final performance on the unseen test set.

In [ ]:
print("Evaluating on Test Set...")
test_scores, test_labels = evaluate_anomaly_scores(model, test_loader)

# Apply Threshold
test_preds = (test_scores > best_threshold).astype(int)

print("\n=== Test Set Results ===")
print(classification_report(test_labels, test_preds, digits=4))

test_pr_auc = auc(*precision_recall_curve(test_labels, test_scores)[1::-1])
test_roc_auc = roc_auc_score(test_labels, test_scores)
print(f"Test PR-AUC:  {test_pr_auc:.4f}")
print(f"Test ROC-AUC: {test_roc_auc:.4f}")

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
sns.histplot(test_scores[test_labels == 0], color='blue', label='Normal', stat='density', alpha=0.5, bins=50)
sns.histplot(test_scores[test_labels == 1], color='red', label='Anomaly', stat='density', alpha=0.5, bins=50)
plt.axvline(best_threshold, color='black', linestyle='--', label='Threshold')
plt.title('Anomaly Scores Distribution')
plt.xlabel('Reconstruction Error')
plt.legend()

plt.subplot(1, 2, 2)
cm = confusion_matrix(test_labels, test_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')

plt.tight_layout()
plt.show()

## 6b. Load a Previously Trained Model (Optional)

Run this cell instead of Section 4 (Training) if you want to load a previously saved model checkpoint. Requires Sections 1-3 to have been executed first (config, data loaders, model class).

In [ ]:
# # ── Load a saved model checkpoint ─────────────────────────────────────────────
# MODEL_DIR = Path("../models/")
# MODEL_PATH = MODEL_DIR / f"{RUN_TAG}_attribute_gae.pt"

# print(f"Loading model from {MODEL_PATH}...")
# checkpoint = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)

# # Reconstruct model with saved architecture dims
# model = AttributeAwareGAE(
#     node_dim=checkpoint['node_dim'],
#     edge_dim=checkpoint['edge_dim'],
#     hidden_dim=checkpoint['hidden_dim'],
#     latent_dim=checkpoint['latent_dim']
# ).to(DEVICE)

# model.load_state_dict(checkpoint['model_state_dict'])
# model.eval()

# best_threshold = checkpoint['best_threshold']
# history = checkpoint.get('history', None)

# print(f"Model loaded successfully.")
# print(f"  Architecture: node_dim={checkpoint['node_dim']}, edge_dim={checkpoint['edge_dim']}, "
#       f"hidden_dim={checkpoint['hidden_dim']}, latent_dim={checkpoint['latent_dim']}")
# print(f"  Best Threshold: {best_threshold:.4f}")
# print(f"  Training history: {'available' if history else 'not saved in this checkpoint'}")

## 7. Training Loss Curves

Visualize how each component of the multi-task loss evolved during training. A healthy model shows all three components decreasing and plateauing.

In [ ]:
if history is None:
    print('No training history available. Train the model (Section 4) or load a checkpoint that includes history.')
else:
  fig, axes = plt.subplots(1, 3, figsize=(18, 5))
  epochs_range = range(1, len(history['total']) + 1)

  # Total Loss
  axes[0].plot(epochs_range, history['total'], 'k-', linewidth=2, label='Total')
  axes[0].set_title('Total Training Loss')
  axes[0].set_xlabel('Epoch')
  axes[0].set_ylabel('Loss')
  axes[0].legend()
  axes[0].grid(True, alpha=0.3)

  # Component Losses (Absolute)
  axes[1].plot(epochs_range, history['structure'], 'b-', linewidth=2, label='Structure (BCE)')
  axes[1].plot(epochs_range, history['node'], 'g-', linewidth=2, label='Node Features (MSE)')
  axes[1].plot(epochs_range, history['edge'], 'r-', linewidth=2, label='Edge Attributes (MSE)')
  axes[1].set_title('Component Losses (Absolute)')
  axes[1].set_xlabel('Epoch')
  axes[1].set_ylabel('Loss')
  axes[1].legend()
  axes[1].grid(True, alpha=0.3)

  # Component Losses (Proportion of Total)
  total = np.array(history['total'])
  str_pct = np.array(history['structure']) / total * 100
  node_pct = np.array(history['node']) / total * 100
  edge_pct = np.array(history['edge']) / total * 100
  axes[2].stackplot(epochs_range, str_pct, node_pct, edge_pct,
                    labels=['Structure', 'Node Features', 'Edge Attributes'],
                    colors=['#4C72B0', '#55A868', '#C44E52'], alpha=0.8)
  axes[2].set_title('Loss Component Proportion (%)')
  axes[2].set_xlabel('Epoch')
  axes[2].set_ylabel('% of Total Loss')
  axes[2].set_ylim(0, 100)
  axes[2].legend(loc='center right')
  axes[2].grid(True, alpha=0.3)

  plt.tight_layout()
  plt.show()

## 8. Component Decomposition of Anomaly Scores

Break down the anomaly score into its three components (Structure, Node, Edge) to understand which signal is most discriminative for separating normal from anomalous graphs.

In [ ]:
def evaluate_component_scores(model, loader):
    """Return per-graph scores decomposed into structure, node, and edge components."""
    model.eval()
    all_str, all_node, all_edge, all_labels = [], [], [], []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc='Component Scoring', leave=False):
            batch = batch.to(DEVICE)
            z, x_norm, edge_attr_norm = model(batch.x, batch.edge_index, batch.edge_attr)
            num_graphs = batch.num_graphs if hasattr(batch, 'num_graphs') else 1
            
            # Structure
            pos_logits = model.decode_structure(z, batch.edge_index)
            pos_probs = torch.sigmoid(pos_logits)
            edge_errors = F.binary_cross_entropy(pos_probs, torch.ones_like(pos_probs), reduction='none')
            edge_batch = batch.batch[batch.edge_index[0]]
            g_str = scatter(edge_errors, edge_batch, dim=0, reduce='mean', dim_size=num_graphs)
            
            # Node
            x_rec = model.decode_node_features(z)
            node_errors = F.mse_loss(x_rec, x_norm, reduction='none').mean(dim=1)
            g_node = scatter(node_errors, batch.batch, dim=0, reduce='mean', dim_size=num_graphs)
            
            # Edge
            if edge_attr_norm is not None and edge_attr_norm.size(0) > 0:
                edge_rec = model.decode_edge_attributes(z, batch.edge_index)
                ea_errors = F.mse_loss(edge_rec, edge_attr_norm, reduction='none').mean(dim=1)
                g_edge = scatter(ea_errors, edge_batch, dim=0, reduce='mean', dim_size=num_graphs)
            else:
                g_edge = torch.zeros_like(g_str)
            
            all_str.append(torch.nan_to_num(g_str, 0.0).cpu())
            all_node.append(g_node.cpu())
            all_edge.append(torch.nan_to_num(g_edge, 0.0).cpu())
            all_labels.append(batch.y.cpu())
            
    return (torch.cat(all_str).numpy(), torch.cat(all_node).numpy(),
            torch.cat(all_edge).numpy(), torch.cat(all_labels).numpy())

test_str, test_node, test_edge, test_y = evaluate_component_scores(model, test_loader)
print(f'Component scores computed for {len(test_y)} test graphs.')

In [ ]:
# --- 8a. Per-Component Score Distributions (Normal vs Anomaly) ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
components = [('Structure Error', test_str), ('Node Feature Error', test_node), ('Edge Attribute Error', test_edge)]
colors = ['#4C72B0', '#55A868', '#C44E52']

for ax, (name, scores), color in zip(axes, components, colors):
    normal_scores = scores[test_y == 0]
    anomaly_scores = scores[test_y == 1]
    
    sns.histplot(normal_scores, color='steelblue', label=f'Normal (n={len(normal_scores)})', stat='density', alpha=0.5, bins=50, ax=ax)
    sns.histplot(anomaly_scores, color='firebrick', label=f'Anomaly (n={len(anomaly_scores)})', stat='density', alpha=0.5, bins=50, ax=ax)
    ax.set_title(name)
    ax.set_xlabel('Reconstruction Error')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Per-Component Score Distributions: Normal vs Anomaly', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- 8b. Component Significance: Per-Component PR-AUC & ROC-AUC ---
from sklearn.metrics import average_precision_score

component_names = ['Structure', 'Node Features', 'Edge Attributes', 'Combined (All)']
component_scores = [test_str, test_node, test_edge, test_str + test_node + test_edge]

pr_aucs, roc_aucs = [], []
for scores in component_scores:
    pr_aucs.append(average_precision_score(test_y, scores))
    roc_aucs.append(roc_auc_score(test_y, scores))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
bar_colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']
x_pos = range(len(component_names))

bars1 = axes[0].bar(x_pos, pr_aucs, color=bar_colors, alpha=0.85, edgecolor='black')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(component_names, rotation=15, ha='right')
axes[0].set_title('PR-AUC by Scoring Component')
axes[0].set_ylabel('PR-AUC')
axes[0].set_ylim(0, 1.05)
axes[0].grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars1, pr_aucs):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02, f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')

bars2 = axes[1].bar(x_pos, roc_aucs, color=bar_colors, alpha=0.85, edgecolor='black')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(component_names, rotation=15, ha='right')
axes[1].set_title('ROC-AUC by Scoring Component')
axes[1].set_ylabel('ROC-AUC')
axes[1].set_ylim(0, 1.05)
axes[1].grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars2, roc_aucs):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02, f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')

plt.suptitle('Component Significance for Anomaly Detection', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Print a summary table
print(f"{'Component':<22} {'PR-AUC':>8} {'ROC-AUC':>8}")
print('-' * 40)
for name, pr, roc in zip(component_names, pr_aucs, roc_aucs):
    print(f'{name:<22} {pr:>8.4f} {roc:>8.4f}')

## 9. Precision-Recall & ROC Curves

In [ ]:
from sklearn.metrics import roc_curve

combined_scores = test_str + test_node + test_edge

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PR Curve
prec, rec, _ = precision_recall_curve(test_y, combined_scores)
pr_auc_val = auc(rec, prec)
axes[0].plot(rec, prec, 'b-', linewidth=2, label=f'PR Curve (AUC = {pr_auc_val:.4f})')
axes[0].fill_between(rec, prec, alpha=0.1, color='blue')
# Baseline: fraction of positives
baseline = test_y.sum() / len(test_y)
axes[0].axhline(baseline, color='gray', linestyle='--', label=f'Random Baseline ({baseline:.3f})')
axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall Curve')
axes[0].legend(loc='lower left')
axes[0].set_xlim([0, 1])
axes[0].set_ylim([0, 1.05])
axes[0].grid(True, alpha=0.3)

# ROC Curve
fpr, tpr, _ = roc_curve(test_y, combined_scores)
roc_auc_val = auc(fpr, tpr)
axes[1].plot(fpr, tpr, 'r-', linewidth=2, label=f'ROC Curve (AUC = {roc_auc_val:.4f})')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='red')
axes[1].plot([0, 1], [0, 1], 'gray', linestyle='--', label='Random Baseline')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend(loc='lower right')
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1.05])
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Component Contribution Analysis

For each anomalous graph that was correctly detected, calculate which reconstruction error (Structure, Node, Edge) contributed the most to the anomaly score. This reveals the dominant signal for each detected anomaly.

In [ ]:
# Identify True Positives (correctly detected anomalies)
tp_mask = (test_y == 1) & (test_preds == 1)
fn_mask = (test_y == 1) & (test_preds == 0)

# For True Positives: which component contributed the most?
tp_str  = test_str[tp_mask]
tp_node = test_node[tp_mask]
tp_edge = test_edge[tp_mask]

tp_total = tp_str + tp_node + tp_edge
tp_str_pct  = (tp_str  / tp_total) * 100
tp_node_pct = (tp_node / tp_total) * 100
tp_edge_pct = (tp_edge / tp_total) * 100

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 10a. Average contribution of each component to detected anomalies
avg_contributions = [tp_str_pct.mean(), tp_node_pct.mean(), tp_edge_pct.mean()]
wedge_colors = ['#4C72B0', '#55A868', '#C44E52']
axes[0].pie(avg_contributions, labels=['Structure', 'Node Features', 'Edge Attributes'],
            autopct='%1.1f%%', colors=wedge_colors, startangle=90, textprops={'fontsize': 11})
axes[0].set_title(f'Avg. Contribution to Detected Anomalies\n(n={tp_mask.sum()})')

# 10b. Per-anomaly dominant component
dominant = np.argmax(np.column_stack([tp_str, tp_node, tp_edge]), axis=1)
dominant_labels = ['Structure', 'Node Features', 'Edge Attributes']
dominant_counts = [np.sum(dominant == i) for i in range(3)]
axes[1].bar(dominant_labels, dominant_counts, color=wedge_colors, edgecolor='black', alpha=0.85)
axes[1].set_title(f'Dominant Component per Detected Anomaly\n(n={tp_mask.sum()})')
axes[1].set_ylabel('Count')
axes[1].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(dominant_counts):
    axes[1].text(i, v + 0.5, str(v), ha='center', fontweight='bold')

# 10c. Box plot: component scores for Normal vs Anomaly
df_box = pd.DataFrame({
    'Structure': np.concatenate([test_str[test_y == 0], test_str[test_y == 1]]),
    'Node Features': np.concatenate([test_node[test_y == 0], test_node[test_y == 1]]),
    'Edge Attributes': np.concatenate([test_edge[test_y == 0], test_edge[test_y == 1]]),
    'Label': ['Normal'] * (test_y == 0).sum() + ['Anomaly'] * (test_y == 1).sum()
})
df_melted = df_box.melt(id_vars='Label', var_name='Component', value_name='Error')
sns.boxplot(data=df_melted, x='Component', y='Error', hue='Label', ax=axes[2],
            palette={'Normal': 'steelblue', 'Anomaly': 'firebrick'})
axes[2].set_title('Error Distribution by Component & Label')
axes[2].set_ylabel('Reconstruction Error')
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Summary Statistics
print(f"\n--- Detected Anomalies (True Positives): {tp_mask.sum()} ---")
print(f"  Avg Structure Error:  {tp_str.mean():.4f} ({tp_str_pct.mean():.1f}% of total)")
print(f"  Avg Node Error:       {tp_node.mean():.4f} ({tp_node_pct.mean():.1f}% of total)")
print(f"  Avg Edge Error:       {tp_edge.mean():.4f} ({tp_edge_pct.mean():.1f}% of total)")
print(f"\n--- Missed Anomalies (False Negatives): {fn_mask.sum()} ---")
if fn_mask.sum() > 0:
    fn_total = test_str[fn_mask] + test_node[fn_mask] + test_edge[fn_mask]
    print(f"  Avg Total Score: {fn_total.mean():.4f} (threshold: {best_threshold:.4f})")
    print(f"  Avg Structure Error:  {test_str[fn_mask].mean():.4f}")
    print(f"  Avg Node Error:       {test_node[fn_mask].mean():.4f}")
    print(f"  Avg Edge Error:       {test_edge[fn_mask].mean():.4f}")
else:
    print('  None! All anomalies were detected.')

## 11. Data Leakage Verification

Verify that no information from the validation or test sets leaked into the training process.

In [ ]:
# --- 11a. Split Overlap Check ---
train_bids = set(g.block_id for g in train_graphs)
val_bids   = set(g.block_id for g in val_graphs)
test_bids  = set(g.block_id for g in test_graphs)

overlap_tv = train_bids & val_bids
overlap_tt = train_bids & test_bids
overlap_vt = val_bids & test_bids

print('=== Data Leakage Verification ===')
print(f'Train block_ids: {len(train_bids):,}')
print(f'Val   block_ids: {len(val_bids):,}')
print(f'Test  block_ids: {len(test_bids):,}')
print(f'\nOverlap Train-Val:  {len(overlap_tv)} {"PASS" if len(overlap_tv) == 0 else "FAIL"}')
print(f'Overlap Train-Test: {len(overlap_tt)} {"PASS" if len(overlap_tt) == 0 else "FAIL"}')
print(f'Overlap Val-Test:   {len(overlap_vt)} {"PASS" if len(overlap_vt) == 0 else "FAIL"}')

# --- 11b. Label Distribution Check ---
train_labels = np.array([g.y.item() for g in train_graphs])
val_labels_check = np.array([g.y.item() for g in val_graphs])
test_labels_check = np.array([g.y.item() for g in test_graphs])

print(f'\n--- Label Distribution ---')
print(f'Train: {len(train_labels):,} graphs, anomaly rate: {train_labels.mean():.4f} ({train_labels.sum():,} anomalies)')
print(f'Val:   {len(val_labels_check):,} graphs, anomaly rate: {val_labels_check.mean():.4f} ({val_labels_check.sum():,} anomalies)')
print(f'Test:  {len(test_labels_check):,} graphs, anomaly rate: {test_labels_check.mean():.4f} ({test_labels_check.sum():,} anomalies)')

# --- 11c. Training Label Usage Check ---
if TRAIN_MODE == 'clean':
    assert train_labels.sum() == 0, 'FAIL: Anomalous graphs found in clean training set!'
    print(f'\nClean-Train Mode: Training set contains 0 anomalies. PASS')
    print('Labels were used ONLY for filtering the training set (not passed to the loss function).')
else:
    print(f'\nNoisy-Train Mode: Labels were never used during training. PASS')

# --- 11d. BatchNorm Running Stats Check ---
print(f'\n--- BatchNorm Running Statistics ---')
print(f'raw_node_norm running_mean sample (first 5 dims): {model.raw_node_norm.running_mean[:5].cpu().numpy()}')
print(f'raw_node_norm running_var  sample (first 5 dims): {model.raw_node_norm.running_var[:5].cpu().numpy()}')
print(f'Note: These statistics were updated ONLY during training (model.train()).')
print(f'During evaluation (model.eval()), running stats are frozen and reused. PASS')

## 12. Save Model

In [ ]:
# Define the path where the model will be saved
MODEL_DIR = Path("../models/")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODEL_DIR / f"{RUN_TAG}_attribute_gae.pt"

# Save the state dictionary AND the architectural dimensions
checkpoint = {
    "node_dim": NODE_DIM,
    "edge_dim": EDGE_DIM,
    "hidden_dim": HIDDEN_DIM,
    "latent_dim": LATENT_DIM,
    "model_state_dict": model.state_dict(),
    "best_threshold": best_threshold,
    "history": history if 'history' in dir() else None,  # Save training curves if available
}

torch.save(checkpoint, MODEL_PATH)
print(f"Model successfully saved to {MODEL_PATH}")